In [8]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
from tensorflow import cast, float32,reduce_mean,maximum
pd.set_option('future.no_silent_downcasting', True)
import numpy as np
import joblib
import gc
inst = joblib.load("../scalers/instituicoes_validas.joblib")
from keras.models import load_model
import matplotlib.pyplot as plt
from funcoes_de_treinamento import * 


In [ ]:

dim_keys = ["Horario", "Diario", "10minutos"]
model_keys = ["GRU", "LSTM", "RNN"]

modelos ={
    "GRU":{
        "Diario": load_model('../../MODELOS/gru_diaria_otimizada_sem_id.keras', compile=False),
        "Horario": load_model('../../MODELOS/gru_horaria_otimizada_sem_id.keras', compile=False),
        "10minutos": load_model('../../MODELOS/gru_10min_otimizada_sem_id.keras', compile=False)
    },
    "LSTM":{
        "Diario": load_model('../../MODELOS/lstm_diaria_otimizada_sem_id.keras', compile=False),
        "Horario": load_model('../../MODELOS/lstm_horaria_otimizada_sem_id.keras', compile=False),
        "10minutos": load_model('../../MODELOS/lstm_10min_otimizada_sem_id.keras', compile=False)
    },
    "RNN":{
        "Diario": load_model('../../MODELOS/rnn_diaria_otimizada_sem_id.keras', compile=False),
        "Horario": load_model('../../MODELOS/rnn_horaria_otimizada_sem_id.keras', compile=False),
        "10minutos": load_model('../../MODELOS/rnn_10min_otimizada_sem_id.keras', compile=False)
    }
}

In [ ]:


train_day_wind = pd.read_csv('../../data/Tabelas_criadas/treino_dia.csv')
test_day_wind = pd.read_csv('../../data/Tabelas_criadas/teste_dia.csv')
val_day_wind = pd.read_csv('../../data/Tabelas_criadas/val_dia.csv')
inputs_day = 7
outputs_day = 1


np_train_d = np.array(train_day_wind)
np_test_d = np.array(test_day_wind)
np_val_d = np.array(val_day_wind)
del train_day_wind, test_day_wind, val_day_wind
X_train_d = np_train_d[:, :inputs_day + 1].astype('float32')
y_train_d = np_train_d[:, inputs_day+1:].astype('float32')
del np_train_d
X_test_d = np_test_d[:, :inputs_day + 1].astype('float32')
y_test_d = np_test_d[:, np.r_[inputs_day+1:inputs_day+1+y_train_d.shape[1],0]].astype('float32')
del np_test_d
X_val_d = np_val_d[:, :inputs_day + 1].astype('float32')
y_val_d = np_val_d[:, inputs_day+1:].astype('float32')
del np_val_d

gc.collect()

In [ ]:
train_hour_wind = pd.read_csv('../../data/Tabelas_criadas/treino_hora.csv')
test_hour_wind = pd.read_csv('../../data/Tabelas_criadas/teste_hora.csv')
val_hour_wind = pd.read_csv('../../data/Tabelas_criadas/val_hora.csv')
inputs_hour = 7*24
outputs_hour = 1*24

np_train_h = np.array(train_hour_wind)
np_test_h = np.array(test_hour_wind)
np_val_h = np.array(val_hour_wind)
del train_hour_wind, test_hour_wind, val_hour_wind
X_train_h = np_train_h[:, :inputs_hour + 1].astype('float32')
y_train_h = np_train_h[:, inputs_hour+1:].astype('float32')
del np_train_h
X_test_h = np_test_h[:, :inputs_hour + 1].astype('float32')
y_test_h = np_test_h[:, np.r_[inputs_hour+1:inputs_hour+1+y_train_h.shape[1],0]].astype('float32')
del np_test_h
X_val_h = np_val_h[:, :inputs_hour + 1].astype('float32')
y_val_h = np_val_h[:, inputs_hour+1:].astype('float32')
del np_val_h

gc.collect()

In [ ]:
train_10min_wind = pd.read_csv('../../data/Tabelas_criadas/treino_10min.csv')
test_10min_wind = pd.read_csv('../../data/Tabelas_criadas/teste_10min.csv')
val_10min_wind = pd.read_csv('../../data/Tabelas_criadas/val_10min.csv')



inputs_10m = 7*24*6
outputs_10m = 1*24*6

np_train_10m = np.array(train_10min_wind)
np_test_10m = np.array(test_10min_wind)
np_val_10m = np.array(val_10min_wind)
del train_10min_wind, test_10min_wind, val_10min_wind
X_train_10m = np_train_10m[:, :inputs_10m + 1].astype('float32')
y_train_10m = np_train_10m[:, inputs_10m+1:].astype('float32')
del np_train_10m
X_test_10m = np_test_10m[:, :inputs_10m + 1].astype('float32')
y_test_10m= np_test_10m[:, np.r_[inputs_10m+1:inputs_10m+1+y_train_10m.shape[1],0]].astype('float32')
del np_test_10m
X_val_10m = np_val_10m[:, :inputs_10m + 1].astype('float32')
y_val_10m = np_val_10m[:, inputs_10m+1:].astype('float32')
del np_val_10m



gc.collect()

In [ ]:
from keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

def avaliar_modelo(y_real, y_previsto, verbose = False):
    mse = mean_squared_error(y_real, y_previsto)
    rmse = math.sqrt(mse)
    nrmse = rmse / (max(y_real) - min(y_real))
    mae = mean_absolute_error(y_real, y_previsto)
    _smape = smape(y_real, y_previsto) 
    if verbose:
        print(f"--- Desempenho: ---")
        print(f"RMSE (Erro Médio): {rmse:.4f}")
        print(f"MAE  (Erro Absoluto): {mae:.4f}")
        print(f"SMAPE: {_smape:.4f}")
        print(f"NRMSE: {nrmse:.4f}")
        print("-" * 30)
    return { "RMSE":rmse,"MAE": mae, "NRMSE": nrmse, "SMAPE": _smape}

def comparar_desempeho_granularidade(X_test_d, X_test_h, X_test_10m, y_test_d, y_test_h, y_test_10m, MODELO_d, MODELO_h,MODELO_10MIN):
    print("Carregando modelos...")
    y_pred_d = MODELO_d.predict(X_test_d)
    y_pred_h = MODELO_h.predict(X_test_h)
    y_pred_10m = MODELO_10MIN.predict(X_test_10m)
    
    print(f"Desempenho do modelo para granularidade diária:")
    resultado_d = avaliar_modelo(y_test_d.flatten(), y_pred_d.flatten(), verbose=True)
    
    print(f"Desempenho do modelo para granularidade horária:")
    resultado_h = avaliar_modelo(y_test_h.flatten(), y_pred_h.flatten(), verbose=True)

    print(f"Desempenho do modelo para granularidade 10minutos:")
    resultado_10m = avaliar_modelo(y_test_10m.flatten(), y_pred_10m.flatten(), verbose=True)    
    
    return {"Diario": resultado_d, "Horario": resultado_h, "10minutos:": resultado_10m}

def separar_dados_por_instituicao(inst, X_test, y_test =  None):
    idx = np.where(X_test[:,0] == inst)[0]
    X_test_i = X_test[idx]
    if y_test is not None:
        y_test_i = y_test[idx]
        return X_test_i, y_test_i
    else:
        return X_test_i
def avaliar_modelo_inst(inst:list,X_test_d, X_test_h, y_test_d, y_test_h, modelo_d, modelo_h):
    resultados = {
        "instituição": [],
        "granularidade:": [],
        "MAE": [],
        "RMSE": [],
        "NRMSE": [],
        "SMAPE": []
    }
    for i in inst:
        print(f"\n##############################\n \
Avaliando instituição {i}... \
                \n##############################\n")
        X_test_d_i, y_test_d_i = separar_dados_por_instituicao(i, X_test_d, y_test_d)
        X_test_h_i, y_test_h_i = separar_dados_por_instituicao(i, X_test_h, y_test_h)
        resultado = comparar_desempeho_granularidade(X_test_d_i, X_test_h_i, y_test_d_i, y_test_h_i, modelo_d, modelo_h)
        resultados["instituição"].append(i)
        resultados["granularidade:"].append("diária")
        resultados["MAE"].append(resultado["Diario"]["MAE"])
        resultados["RMSE"].append(resultado["Diario"]["RMSE"])
        resultados["NRMSE"].append(resultado["Diario"]["NRMSE"])
        resultados["SMAPE"].append(resultado["Diario"]["SMAPE"])

        resultados["instituição"].append(i)
        resultados["granularidade:"].append("horária")
        resultados["MAE"].append(resultado["Horario"]["MAE"])
        resultados["RMSE"].append(resultado["Horario"]["RMSE"])
        resultados["NRMSE"].append(resultado["Horario"]["NRMSE"])
        resultados["SMAPE"].append(resultado["Horario"]["SMAPE"])
    return pd.DataFrame(resultados)



        
        



In [ ]:
metricas = {}
for model in model_keys:
    print(model)
    metricas[model] = comparar_desempeho_granularidade(
        X_test_d[:,1:].reshape((X_test_d.shape[0], 1, X_test_d.shape[1]-1)), 
        X_test_h[:,1:].reshape((X_test_h.shape[0], 1, X_test_h.shape[1]-1)),
        X_test_10m[:,1:].reshape((X_test_10m.shape[0], 1, X_test_10m.shape[1]-1)), 
        y_test_d[:,0], 
        y_test_h[:,:24],
        y_test_10m[:,:24*6],
        modelos[model]["Diario"],
        modelos[model]["Horario"],
        modelos[model]["10minutos"]
    )

In [ ]:

predicoes = {}
predicoes_sem_escaler = {}
real = {
    "Diario": y_test_d[:,0],
    "Horario": y_test_h[:,:24],
    "10minutos": y_test_10m[:,:24*6]
}
for model in model_keys:
    print(model)
    predicoes[model] = {
        "Diario": modelos[model]["Diario"].predict(X_test_d[:,1:].reshape((X_test_d.shape[0], 1, X_test_d.shape[1]-1))),
        "Horario": modelos[model]["Horario"].predict(X_test_h[:,1:].reshape((X_test_h.shape[0], 1, X_test_h.shape[1]-1))),
        "10minutos": modelos[model]["10minutos"].predict(X_test_10m[:,1:].reshape((X_test_10m.shape[0], 1, X_test_10m.shape[1]-1)))
    }
        

### Desescalonando dias preditos

In [2]:
scalers_day = joblib.load('../scalers/scalers_day.joblib')
real_sem_escaler = {}
predicoes_sem_escaler = {}
for model in model_keys:
    predicoes_sem_escaler[model] = {}
    predicoes_sem_escaler[model]["Diario"] = []
    real_sem_escaler["Diario"] = []
    df_d = pd.DataFrame({
            "previsto": predicoes[model]["Diario"].flatten()
        })
    df_d_real = pd.DataFrame({
            "real": real["Diario"].flatten()
        })
    for i in inst:
        df_d["i"] = X_test_d[:,0]
        df_d_real["i"] = X_test_d[:,0]
        
        df_d_i = df_d[df_d["i"] == int(i)]
        sem_scaler = scalers_day[i].inverse_transform(df_d_i["previsto"].values.reshape(-1, 1))
        sem_scaler_real = scalers_day[i].inverse_transform(df_d_real[df_d_real["i"] == int(i)]["real"].values.reshape(-1, 1))
        predicoes_sem_escaler[model]["Diario"].extend(sem_scaler.flatten()) 
        real_sem_escaler["Diario"].extend(sem_scaler_real.flatten())

    predicoes_sem_escaler[model]["Diario"] = np.array(predicoes_sem_escaler[model]["Diario"])
    real_sem_escaler["Diario"] = np.array(real_sem_escaler["Diario"])
        

NameError: name 'model_keys' is not defined

### Desescalonando Horário

In [3]:
import numpy as np
scalers_hour = joblib.load('../scalers/scalers_hour.joblib')
ids_instituicoes = X_test_h[:, 0]
instituicoes_unicas = np.unique(ids_instituicoes)

for model in model_keys:
    preds_originais = predicoes[model]["Horario"]
    real_originais = real["Horario"]
    
    # Listas temporárias para guardar os dados limpos e as posições originais deles
    linhas_revertidas = []
    indices_originais = []
    linhas_revertidas_real = []
    indices_originais_real = []

    for i in instituicoes_unicas:
        # Pega as posições exatas (números das linhas) onde esta instituição aparece na matriz original
        indices_inst = np.where(ids_instituicoes == i)[0]

        # Aplica o salto de 24 em 24 apenas nesses índices
        indices_filtrados = indices_inst

        if len(indices_filtrados) > 0:
            scaler = scalers_hour[str(int(i))]
            
            # Puxa os dados da matriz original usando os índices já filtrados
            dados_inst_filtrado = preds_originais[indices_filtrados]
            real_inst_filtrado = real_originais[indices_filtrados]

            # Achata, desnormaliza e devolve para o formato de colunas
            dados_achatados = dados_inst_filtrado.reshape(-1, 1)
            dados_revertidos = scaler.inverse_transform(dados_achatados).reshape(dados_inst_filtrado.shape)
            real_achatados = real_inst_filtrado.reshape(-1, 1)
            real_revertidos = scaler.inverse_transform(real_achatados).reshape(real_inst_filtrado.shape)

            # Guarda os dados revertidos e anota a posição original de cada um
            linhas_revertidas.extend(dados_revertidos)
            indices_originais.extend(indices_filtrados)
            linhas_revertidas_real.extend(real_revertidos)
            indices_originais_real.extend(indices_filtrados)

    # Converte as listas de volta para matrizes do NumPy
    linhas_revertidas = np.array(linhas_revertidas)
    indices_originais = np.array(indices_originais)
    linhas_revertidas_real = np.array(linhas_revertidas_real)
    indices_originais_real = np.array(indices_originais_real)

    # O PASSO FUNDAMENTAL: Descobre a ordem correta para reembaralhar os dados de volta ao formato cronológico
    ordem_correta = np.argsort(indices_originais)
    
    # Salva a matriz perfeita, desnormalizada, filtrada e ordenada
    predicoes_sem_escaler[model]["Horario"] = linhas_revertidas[ordem_correta]
    real_sem_escaler["Horario"] = linhas_revertidas_real[np.argsort(indices_originais_real)]

NameError: name 'X_test_h' is not defined

### Desescalonando minutos

In [ ]:
import numpy as np
scalers_10min = joblib.load('../scalers/scalers_10min.joblib')
ids_instituicoes = X_test_10m[:, 0]
instituicoes_unicas = np.unique(ids_instituicoes)

for model in model_keys:
    preds_originais = predicoes[model]["10minutos"]
    real_originais = real["10minutos"]
    
    # Listas temporárias para guardar os dados limpos e as posições originais deles
    linhas_revertidas = []
    indices_originais = []
    linhas_revertidas_real = []
    indices_originais_real = []

    for i in instituicoes_unicas:
        # Pega as posições exatas (números das linhas) onde esta instituição aparece na matriz original
        indices_inst = np.where(ids_instituicoes == i)[0]

        # Aplica o salto de 24 em 24 apenas nesses índices
        indices_filtrados = indices_inst

        if len(indices_filtrados) > 0:
            scaler = scalers_10min[str(int(i))]
            
            # Puxa os dados da matriz original usando os índices já filtrados
            dados_inst_filtrado = preds_originais[indices_filtrados]
            real_inst_filtrado = real_originais[indices_filtrados]

            # Achata, desnormaliza e devolve para o formato de colunas
            dados_achatados = dados_inst_filtrado.reshape(-1, 1)
            dados_revertidos = scaler.inverse_transform(dados_achatados).reshape(dados_inst_filtrado.shape)
            real_achatados = real_inst_filtrado.reshape(-1, 1)
            real_revertidos = scaler.inverse_transform(real_achatados).reshape(real_inst_filtrado.shape)

            # Guarda os dados revertidos e anota a posição original de cada um
            linhas_revertidas.extend(dados_revertidos)
            indices_originais.extend(indices_filtrados)
            linhas_revertidas_real.extend(real_revertidos)
            indices_originais_real.extend(indices_filtrados)

    # Converte as listas de volta para matrizes do NumPy
    linhas_revertidas = np.array(linhas_revertidas)
    indices_originais = np.array(indices_originais)
    linhas_revertidas_real = np.array(linhas_revertidas_real)
    indices_originais_real = np.array(indices_originais_real)

    # O PASSO FUNDAMENTAL: Descobre a ordem correta para reembaralhar os dados de volta ao formato cronológico
    ordem_correta = np.argsort(indices_originais)
    
    # Salva a matriz perfeita, desnormalizada, filtrada e ordenada
    predicoes_sem_escaler[model]["10minutos"] = linhas_revertidas[ordem_correta]
    real_sem_escaler["10minutos"] = linhas_revertidas_real[np.argsort(indices_originais_real)]

## Plotando os gráficos

In [ ]:
modelos = list(metricas.keys())
metricas_plot = ['RMSE','MAE', 'NRMSE'] 

# Adição de um terceiro tom (mais claro) para a granularidade de 10 minutos


cores = {
    'RMSE': {'Diario': "#155fe9", 'Horario': "#ff3f0f", '10minutos:': "#73ff00"},
    'MAE': {'Diario': '#1f77b4', 'Horario': '#ff7f0e', '10minutos:': '#2ca02c'},
    'NRMSE': {'Diario': '#aec7e8', 'Horario': '#ffbb78', '10minutos:': '#98df8a'},
}

# Configuração do espaçamento das 9 barras
x = np.arange(len(modelos))

# A largura da barra foi reduzida para 0.09. 
# Assim, 9 barras ocupam 0.81 do espaço, deixando uma margem limpa entre os modelos.
largura_barra = 0.09  
offsets = [-4, -3, -2, -1, 0, 1, 2, 3, 4] 

fig, ax = plt.subplots(figsize=(16, 8)) # Figura levemente mais larga

idx_offset = 0
for metrica in metricas_plot:
    for granularidade in ['Diario', 'Horario', '10minutos:']:
        valores = [metricas[modelo][granularidade][metrica] for modelo in modelos]
        posicao = x + (offsets[idx_offset] * largura_barra)
        
        # Ajuste dinâmico do nome para a legenda
        if granularidade == "Diario":
            nome_gran = "Diária"
        elif granularidade == "Horario":
            nome_gran = "Horária"
        else:
            nome_gran = "10 Minutos"
            
        label = f'{metrica} ({nome_gran})'
        cor = cores[metrica][granularidade]
        
        barras = ax.bar(posicao, valores, largura_barra, label=label, color=cor)
        
        # Valores no topo
        ax.bar_label(barras, padding=3, fmt='%.4f', fontsize=8, rotation=90)
        idx_offset += 1

ax.set_ylabel('Valor do Erro')
ax.set_title('Comparativo de Desempenho - Diário vs Horário vs 10 Minutos', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(modelos, fontsize=11)

ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True) 

# Ajuste automático e seguro do limite Y para evitar corte dos números rotacionados
valor_maximo = max([metricas[mod][gran][met] for mod in modelos for gran in ['Diario', 'Horario', '10minutos:'] for met in metricas_plot])
ax.set_ylim(0, valor_maximo * 1.3)

ax.legend(title='Métrica e Granularidade', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

granularidade = {
    "Horario": 24,
    "10minutos": 24*6
}

# Define qual granularidade você quer plotar neste gráfico
gran = "10minutos"

# 1. Cria a figura UMA ÚNICA VEZ antes do loop
plt.figure(figsize=(18, 6))

# 2. Loop para cada modelo: calcula os erros e adiciona a linha no gráfico aberto
for modelo in model_keys:
    erros_por_horizonte = []

    # Calcula o erro separadamente para cada passo no futuro
    for t in range(granularidade[gran]):
        real_coluna = real_sem_escaler[gran][:, t]
        pred_coluna = predicoes_sem_escaler[modelo][gran][:, t]

        # Calcula o RMSE dessa hora específica
        rmse_atual = np.sqrt(mean_squared_error(real_coluna, pred_coluna))
        erros_por_horizonte.append(rmse_atual)

    # Plota a linha deste modelo (o parâmetro 'label' garante que o nome vá para a legenda)
    plt.plot(range(1, granularidade[gran] + 1), erros_por_horizonte, marker='o', linewidth=2, label=modelo)

# 3. Estilização do gráfico unificado
plt.title(f'Curva de Degradação Comparativa (Horizonte {gran}) - Dados Reais', fontsize=14, fontweight='bold')
plt.xlabel('Horizonte de Previsão (Passos no futuro)', fontsize=12)
plt.ylabel('Erro RMSE (Bytes)', fontsize=12)

# Configura o eixo X para mostrar todos os passos
plt.xticks(range(1, granularidade[gran] + 1,2), rotation = 75)
plt.grid(True, linestyle='--', alpha=0.6)

# Invoca a legenda para mostrar qual cor pertence a qual modelo
plt.legend(title='Modelos', fontsize=11, loc='upper left', bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.show()